In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
# Path to JAR
import os
credentials_location = os.environ.get(
    "GOOGLE_APPLICATION_CREDENTIALS",
    "/home/your-user/.gcp/service-account.json"
)

In [ ]:

# 1. Create SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("taxi_yellow_green") \
    .config(
        "spark.jars",
        "/ruta/a/gcs-connector-hadoop3-latest.jar"
    ) \
    .config(
        "spark.hadoop.google.cloud.auth.service.account.enable",
        "true"
    ) \
    .config(
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile",
        credentials_location
    ) \
    .getOrCreate()

In [ ]:
# 2. hadoopConfiguration
hadoop_conf = spark._jsc.hadoopConfiguration()

hadoop_conf.set(
    "fs.AbstractFileSystem.gs.impl",
    "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS"
)
hadoop_conf.set(
    "fs.gs.impl",
    "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem"
)
hadoop_conf.set(
    "fs.gs.auth.service.account.json.keyfile",
    credentials_location
)
hadoop_conf.set(
    "fs.gs.auth.service.account.enable",
    "true"
)

In [ ]:
df = spark.read.parquet("gs://data-engineering-demo/yellow/")
df.printSchema()